# 🌸 Iris Dataset — Data Models Training & Testing
---

**Dataset :** Iris Flower Dataset · 150 samples · 3 species · 4 features  
**Libraries:** Seaborn · Matplotlib · Scikit-learn · Pandas · NumPy  
**Models Covered:**
| # | Model | Type |
|---|---|---|
| 1 | Logistic Regression | Linear Classifier |
| 2 | K-Nearest Neighbors (KNN) | Instance-based |
| 3 | Decision Tree | Tree-based |
| 4 | Random Forest | Ensemble |
| 5 | Support Vector Machine (SVM) | Margin-based |
---

* ##  Importing Libraries

We import visualization libraries (**Seaborn**, **Matplotlib**), data-processing libraries (**Pandas**, **NumPy**), and Scikit-learn modules for model building, evaluation, and preprocessing.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing   import LabelEncoder, StandardScaler
from sklearn.metrics         import (accuracy_score, classification_report,
                                     confusion_matrix, ConfusionMatrixDisplay)
from sklearn.linear_model    import LogisticRegression
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.tree            import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble        import RandomForestClassifier
from sklearn.svm             import SVC
from pathlib                 import Path

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 13,
                     "axes.labelsize": 11, "legend.fontsize": 10})

---
* ##  Loading & Preparing the Dataset

We load the Iris CSV, drop the index column, rename features for cleanliness, and strip the `Iris-` prefix from species labels. We then encode the target column into integers (required by most sklearn models).

 Species | Description |
|---|---|
| *Iris-setosa* | Easily separable — small petals |
| *Iris-versicolor* | Overlaps slightly with virginica |
| *Iris-virginica* | Largest petals among the three |

Each sample records **four measurements** (in centimetres):
- Sepal Length & Sepal Width
- Petal Length & Petal Width



In [ ]:
BASE_DIR = Path.cwd().parent
file_path = BASE_DIR / 'data' / 'Iris.csv'

df = pd.read_csv(file_path)
df.drop(columns=['Id'], inplace=True)
df.columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species']
df['species'] = df['species'].str.replace('Iris-', '', regex=False)


le = LabelEncoder()
df['species_enc'] = le.fit_transform(df['species'])   
class_names = le.classes_

print(f"Shape          : {df.shape}")
print(f"Species labels : {class_names.tolist()} → {[0,1,2]}")
df.head(8)

---
* ## Train / Test Split & Feature Scaling

We split the data **80 % training / 20 % test** with stratification so each species is equally represented in both sets. Feature scaling (StandardScaler) is applied for models sensitive to feature magnitude (Logistic Regression, KNN, SVM).


In [ ]:
X = df[['sepal_length','sepal_width','petal_length','petal_width']].values
y = df['species_enc'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler  = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Training samples : {X_train.shape[0]}  (per class: {dict(zip(*np.unique(y_train, return_counts=True)))})")
print(f"Test samples     : {X_test.shape[0]}   (per class: {dict(zip(*np.unique(y_test,  return_counts=True)))})")

---
* ## Reusable Helper Functions

We define a helper to **plot a confusion matrix** and another to **visualise decision boundaries** using the two most discriminative features (petal_length & petal_width). These will be reused for all five models.


In [ ]:
def plot_confusion(y_true, y_pred, model_name, ax=None):
    """Plot a labelled confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    show = ax is None
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 4))
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f"Confusion Matrix — {model_name}", fontweight='bold')
    if show:
        plt.tight_layout(); plt.show()


def plot_decision_boundary(clf, X_sc, y, model_name, feat_idx=(2, 3)):
    """2-D decision boundary using two scaled features."""
    f0, f1 = feat_idx
    feat_names = ['sepal_length','sepal_width','petal_length','petal_width']
    X2 = X_sc[:, [f0, f1]]

    x_min, x_max = X2[:, 0].min() - 0.5, X2[:, 0].max() + 0.5
    y_min, y_max = X2[:, 1].min() - 0.5, X2[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))

    grid = np.zeros((xx.ravel().shape[0], X_sc.shape[1]))
    grid[:, f0] = xx.ravel()
    grid[:, f1] = yy.ravel()

    Z = clf.predict(grid).reshape(xx.shape)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='Set2')
    scatter = ax.scatter(X2[:, 0], X2[:, 1], c=y,
                         cmap='Set2', edgecolors='k', s=60, zorder=3)
    ax.set_xlabel(feat_names[f0] + " (scaled)")
    ax.set_ylabel(feat_names[f1] + " (scaled)")
    ax.set_title(f"Decision Boundary — {model_name}", fontweight='bold')
    legend_labels = [plt.Line2D([0],[0], marker='o', color='w',
                                markerfacecolor=sns.color_palette('Set2')[i],
                                markersize=9, label=class_names[i]) for i in range(3)]
    ax.legend(handles=legend_labels, title='Species')
    plt.tight_layout(); plt.show()

---
* ## Model 1 — Logistic Regression

**Logistic Regression** is a linear classifier that models the probability of each class using the logistic (sigmoid) function. Despite the word "regression", it is used for **classification**. It works best when classes are linearly separable.

**How it works:**
- Fits a hyperplane to separate classes.
- Outputs probabilities via the softmax function (multi-class via OvR or multinomial).
- Regularisation parameter `C` controls overfitting — smaller `C` = stronger regularisation.


In [ ]:
lr = LogisticRegression(C=1.0, max_iter=200, random_state=42)
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)

acc_lr = accuracy_score(y_test, y_pred_lr)
print(f"Logistic Regression — Test Accuracy: {acc_lr*100:.2f}%")
print()
print(classification_report(y_test, y_pred_lr, target_names=class_names))

The classification report shows **precision, recall, and F1-score** for each species:
- **Precision**: Of all predicted as class X, how many were actually X?
- **Recall**: Of all actual class X, how many did we correctly predict?
- **F1-score**: Harmonic mean of precision and recall.


In [ ]:
plot_confusion(y_test, y_pred_lr, "Logistic Regression")

In [ ]:
plot_decision_boundary(lr, X_test_sc, y_test, "Logistic Regression")

The decision boundary is **linear** — straight lines separate the three regions. Logistic Regression works well here because petal features are nearly linearly separable. Setosa (top-left) is always cleanly separated.

In [ ]:
cv_lr = cross_val_score(lr, scaler.transform(X), y, cv=StratifiedKFold(5), scoring='accuracy')
print(f"5-Fold CV Accuracy : {cv_lr.mean()*100:.2f}% ± {cv_lr.std()*100:.2f}%")
print(f"Per-fold scores    : {[f'{s*100:.1f}%' for s in cv_lr]}")